# DSPy tutorial

In [2]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [31]:
import dspy
from dspy.evaluate import Evaluate
import pandas as pd

## Import data

In [7]:
df = pd.read_csv('data/brass_birmingham-selected.csv')
df.head(3)

,url,question,answer,manual quote 1,manual quote 2,manual quote 3,manual quote 4,manual quote 5,manual quote 6,manual quote 7,manual quote 8
0,https://boardgamegeek.com/thread/2096967/doubl...,May I connect another players brewery with two...,"When laying two rail links, each one is consid...","In a single Network action, you may build a ma...",You must consume 1 coal for each\nrail Link bu...,NaN,NaN,NaN,NaN,NaN,NaN
1,https://boardgamegeek.com/thread/2901920/which...,"Since you need coal to build a rail link, and ...","To consume coal, you need to be connected to a...","To consume coal, a rail Link or Industry tile ...",Coal must be consumed from: 1 The closest (few...,2 If you are not connected to an unflipped Coa...,NaN,NaN,NaN,NaN,NaN
2,https://boardgamegeek.com/thread/3087643/cards...,"In a 4 player game, there are 64 action cards....","Yes, the first round of the game has only one ...",Player Area Setup:,8- Draw 8 cards from the Draw Deck; this is yo...,There are exactly 8/9/10 rounds per era in a 4...,"On your turn, perform a total of 2 actions.\nE...",NaN,NaN,NaN,NaN


In [9]:
# keep only rows with quotes
df = df[df['manual quote 1'] > '']
len(df)

180

In [12]:
# join all of the manual quote columns together 
quote_columns = [f'manual quote {i}' for i in range(1, 9)]
df['manual quote'] = df[quote_columns].apply(
    lambda row: '\n\n'.join(row.dropna().astype(str)),
    axis=1
)
df.head(3)

,url,question,answer,manual quote 1,manual quote 2,manual quote 3,manual quote 4,manual quote 5,manual quote 6,manual quote 7,manual quote 8,manual quote
0,https://boardgamegeek.com/thread/2096967/doubl...,May I connect another players brewery with two...,"When laying two rail links, each one is consid...","In a single Network action, you may build a ma...",You must consume 1 coal for each\nrail Link bu...,NaN,NaN,NaN,NaN,NaN,NaN,"In a single Network action, you may build a ma..."
1,https://boardgamegeek.com/thread/2901920/which...,"Since you need coal to build a rail link, and ...","To consume coal, you need to be connected to a...","To consume coal, a rail Link or Industry tile ...",Coal must be consumed from: 1 The closest (few...,2 If you are not connected to an unflipped Coa...,NaN,NaN,NaN,NaN,NaN,"To consume coal, a rail Link or Industry tile ..."
2,https://boardgamegeek.com/thread/3087643/cards...,"In a 4 player game, there are 64 action cards....","Yes, the first round of the game has only one ...",Player Area Setup:,8- Draw 8 cards from the Draw Deck; this is yo...,There are exactly 8/9/10 rounds per era in a 4...,"On your turn, perform a total of 2 actions.\nE...",NaN,NaN,NaN,NaN,Player Area Setup:\n\n8- Draw 8 cards from the...


In [13]:
# drop the other columns
df = df.drop(columns=quote_columns)

In [17]:
# turn dataset into dspy examples
examples = []

for index, row in df.iterrows():
    example = dspy.Example(
        question=row['question'],
        context=row['manual quote'],
        answer= row['answer'],
    ).with_inputs('question', 'context')
    examples.append(example)
len(examples)

180

In [55]:
# split into train and test
train = examples[:100]
dev = examples[100:140]
test = examples[140:]
print(len(train), len(dev), len(test))

100 40 40


## Set up DSPy

define signatures and modules

In [22]:
lm = dspy.OpenAI(model="gpt-4o-mini")
dspy.settings.configure(lm=lm)

In [20]:
class GenerateAnswer(dspy.Signature):
    """Answer questions from contexts."""

    context = dspy.InputField(desc="may contain relevant facts")
    question = dspy.InputField()
    answer = dspy.OutputField()

In [27]:
class AnswerGenerator(dspy.Module):
    def __init__(self):
        super().__init__()

        self.generate_answer = dspy.ChainOfThought(GenerateAnswer)
    
    def forward(self, question, context):
        prediction = self.generate_answer(context=context, question=question)
        return dspy.Prediction(context=context, answer=prediction.answer)

In [56]:
# make sure everything works
answer_generator = AnswerGenerator()
prediction = answer_generator(test[0]['question'], test[0]['context'])
print(test[0])
print(prediction)

Example({'question': 'If I overbuild a flipped tile, do I lose out on any points that I would have otherwise scored for that overbuilt tile at the end of the current era? Or do I score the points immediately at the time of overbuilding?', 'context': 'Remove overbuilt Industry tiles from the game, and return them to the box (they will not score VPs). Players do not lose previously gained income or VPs if their Industry tiles are overbuilt.', 'answer': "You lose them. Overbuilt tiles (whether yours or an opponent's) do not score any VP, because they cease to exist."}) (input_keys={'question', 'context'})
Prediction(
    context='Remove overbuilt Industry tiles from the game, and return them to the box (they will not score VPs). Players do not lose previously gained income or VPs if their Industry tiles are overbuilt.',
    answer='You do not score any points for the overbuilt tile at the end of the current era; it is removed and does not contribute to scoring.'
)


In [57]:
lm.inspect_history(n=1)




Answer questions from contexts.

---

Follow the following format.

Context: may contain relevant facts

Question: ${question}

Reasoning: Let's think step by step in order to ${produce the answer}. We ...

Answer: ${answer}

---

Context: Remove overbuilt Industry tiles from the game, and return them to the box (they will not score VPs). Players do not lose previously gained income or VPs if their Industry tiles are overbuilt.

Question: If I overbuild a flipped tile, do I lose out on any points that I would have otherwise scored for that overbuilt tile at the end of the current era? Or do I score the points immediately at the time of overbuilding?

Reasoning: Let's think step by step in order to determine the impact of overbuilding a flipped tile on scoring. The context states that overbuilt Industry tiles are removed from the game and returned to the box, meaning they will not score Victory Points (VPs). It also mentions that players do not lose previously gained income or VPs if

"\n\n\nAnswer questions from contexts.\n\n---\n\nFollow the following format.\n\nContext: may contain relevant facts\n\nQuestion: ${question}\n\nReasoning: Let's think step by step in order to ${produce the answer}. We ...\n\nAnswer: ${answer}\n\n---\n\nContext: Remove overbuilt Industry tiles from the game, and return them to the box (they will not score VPs). Players do not lose previously gained income or VPs if their Industry tiles are overbuilt.\n\nQuestion: If I overbuild a flipped tile, do I lose out on any points that I would have otherwise scored for that overbuilt tile at the end of the current era? Or do I score the points immediately at the time of overbuilding?\n\nReasoning: Let's think step by step in order to\x1b determine the impact of overbuilding a flipped tile on scoring. The context states that overbuilt Industry tiles are removed from the game and returned to the box, meaning they will not score Victory Points (VPs). It also mentions that players do not lose previo

## Define evaluation metrics

In [64]:
# set up metrics
metric_lm = dspy.OpenAI(model='gpt-4o', max_tokens=1000, model_type='chat')

class Assess(dspy.Signature):
    """Assess the quality of an answer along the specified dimension."""
    
    context = dspy.InputField(desc="ignore if N/A")
    assessed_text = dspy.InputField()
    assessment_question = dspy.InputField()
    assessment_answer = dspy.OutputField(desc="Yes or No")

def llm_metric(gold, pred, trace=None):
    question, answer, context = gold.question, gold.answer, gold.context
    predicted_answer = pred.answer

    faithfulness_question = "Is the assessed text grounded in the context? " + \
                            "Say no if it includes significant facts not in the context."
    correctness_question = f"The text above should answer `{question}`. The gold answer is `{answer}'. " + \
                           "Does the assessed text above contain the gold answer?"

    with dspy.context(lm=metric_lm):
        faithfulness_answer = dspy.Predict(Assess)(
            context=context, 
            assessed_text=predicted_answer, 
            assessment_question=faithfulness_question,
        )
        correctness_answer = dspy.Predict(Assess)(
            context='N/A', 
            assessed_text=predicted_answer, 
            assessment_question=correctness_question,
        )
    
    faithfulness_score, correctness_score = (any(w.lower() == 'yes' for w in m.assessment_answer.split()) \
                                             for m in [faithfulness_answer, correctness_answer])

    # print(f"Faithful: {faithfulness_answer.assessment_answer}, score={faithfulness_score}")
    # print(f"Correct: {correctness_answer.assessment_answer}, score={correctness_score}")
    
    score = float(faithfulness_score) + float(correctness_score)

    if trace is not None: 
        return score >= 2
        
    return score / 2.0

In [59]:
gold = test[0]
print(f"question {gold.question}")
print(f"gold answer {gold.answer}")
print(f"pred answer {prediction.answer}")
llm_metric(gold, prediction)

question If I overbuild a flipped tile, do I lose out on any points that I would have otherwise scored for that overbuilt tile at the end of the current era? Or do I score the points immediately at the time of overbuilding?
gold answer You lose them. Overbuilt tiles (whether yours or an opponent's) do not score any VP, because they cease to exist.
pred answer You do not score any points for the overbuilt tile at the end of the current era; it is removed and does not contribute to scoring.
Faithful: Assessment Answer: Yes, score=True
Correct: No, score=False


0.5

In [60]:
metric_lm.inspect_history(n=2)




Assess the quality of an answer along the specified dimension.

---

Follow the following format.

Context: ignore if N/A

Assessed Text: ${assessed_text}

Assessment Question: ${assessment_question}

Assessment Answer: Yes or No

---

Context: Remove overbuilt Industry tiles from the game, and return them to the box (they will not score VPs). Players do not lose previously gained income or VPs if their Industry tiles are overbuilt.

Assessed Text: You do not score any points for the overbuilt tile at the end of the current era; it is removed and does not contribute to scoring.

Assessment Question: Is the assessed text grounded in the context? Say no if it includes significant facts not in the context.

Assessment Answer: Assessment Answer: Yes





Assess the quality of an answer along the specified dimension.

---

Follow the following format.

Context: ignore if N/A

Assessed Text: ${assessed_text}

Assessment Question: ${assessment_question}

Assessment Answer: Yes or No

---



"\n\n\nAssess the quality of an answer along the specified dimension.\n\n---\n\nFollow the following format.\n\nContext: ignore if N/A\n\nAssessed Text: ${assessed_text}\n\nAssessment Question: ${assessment_question}\n\nAssessment Answer: Yes or No\n\n---\n\nContext: Remove overbuilt Industry tiles from the game, and return them to the box (they will not score VPs). Players do not lose previously gained income or VPs if their Industry tiles are overbuilt.\n\nAssessed Text: You do not score any points for the overbuilt tile at the end of the current era; it is removed and does not contribute to scoring.\n\nAssessment Question: Is the assessed text grounded in the context? Say no if it includes significant facts not in the context.\n\nAssessment Answer:\x1b Assessment Answer: Yes\x1b\n\n\n\n\n\nAssess the quality of an answer along the specified dimension.\n\n---\n\nFollow the following format.\n\nContext: ignore if N/A\n\nAssessed Text: ${assessed_text}\n\nAssessment Question: ${assessm

## Evaluate uncompiled prompt

In [61]:
# set up evaluator
evaluator = Evaluate(devset=test, num_threads=1, display_progress=True, display_table=5)

In [52]:
# launch evaluation
evaluator(answer_generator, metric=llm_metric)

  0%|                                                              | 0/40 [00:00<?, ?it/s]Faithful: Assessment Answer: Yes, score=True
Correct: No, score=False
Average Metric: 0.5 / 1  (50.0):   2%|▌                    | 1/40 [00:01<01:13,  1.89s/it]Faithful: No, score=False
Correct: No, score=False
Average Metric: 0.5 / 2  (25.0):   5%|█                    | 2/40 [00:03<01:03,  1.68s/it]Faithful: Assessment Answer: No, score=False
Correct: No, score=False
Average Metric: 0.5 / 3  (16.7):   8%|█▌                   | 3/40 [00:05<01:11,  1.93s/it]Faithful: No, score=False
Correct: No, score=False
Average Metric: 0.5 / 4  (12.5):  10%|██                   | 4/40 [00:08<01:19,  2.20s/it]Faithful: No, score=False
Correct: No, score=False
Average Metric: 0.5 / 5  (10.0):  12%|██▋                  | 5/40 [00:10<01:14,  2.12s/it]Faithful: No, score=False
Correct: No, score=False
Average Metric: 0.5 / 6  (8.3):  15%|███▎                  | 6/40 [00:12<01:15,  2.23s/it]Faithful: No, score=False


,question,example_context,example_answer,pred_context,pred_answer,llm_metric
0,"If I overbuild a flipped tile, do I lose out on any points that I would have otherwise scored for that overbuilt tile at the...","Remove overbuilt Industry tiles from the game, and return them to the box (they will not score VPs). Players do not lose previously gained income...","You lose them. Overbuilt tiles (whether yours or an opponent's) do not score any VP, because they cease to exist.","Remove overbuilt Industry tiles from the game, and return them to the box (they will not score VPs). Players do not lose previously gained income...",You do not score any points for the overbuilt tile at the end of the current era; it is removed and does not contribute to...,✔️ [0.5]
1,"If you have no tiles on the board and build an industry or link, do you still pay the usual costs?","To Perform the Build Action: 3- The cost of the Industry tile you are building is displayed, to the left of its slot, on your...","Yes, of course. That also includes needing a source of coal.","To Perform the Build Action: 3- The cost of the Industry tile you are building is displayed, to the left of its slot, on your...","Yes, you still pay the usual costs.",0.0
2,When performing the double-rail action is the beer paid before or after the connections are made?,"In a single Network action, you may build a maximum of 2 rail Links for £15, if you also consume 1 beer. This beer must...",The rules say the second rail link must be connected to an opponent's beer after it is built if you need to use an opponent's...,"In a single Network action, you may build a maximum of 2 rail Links for £15, if you also consume 1 beer. This beer must...",The beer is consumed before the connections are made.,0.0
3,If someone has placed a coal industry at some city in their first turn and now i build a canal to that same city in...,Two locations are considered “connected” to each other if you can trace a route of Link tiles (owned by any player) from one location to...,"It is allowed. Note that your opponent can always build with a location card, so your opponent can use a location card or scout for...",Two locations are considered “connected” to each other if you can trace a route of Link tiles (owned by any player) from one location to...,"Yes, your actions are allowed. Your opponent can still use the coal industry they built, but they cannot expand their network further if all routes...",0.0
4,What is the point of building in areas not considered for a gameplay (2/3 P) when it is out of bounds?,"At the end of each era, flipped Industry tiles score VPs. When flipped, they have a black top half and a VP icon in the...","They still work as normal, so they still produce resources, income, points and link points. They are just a bit harder to reach, but they...","At the end of each era, flipped Industry tiles score VPs. When flipped, they have a black top half and a VP icon in the...",The point of building in areas not considered for gameplay (2/3 P) when it is out of bounds may be for strategic positioning or future...,0.0


22.5

## Compile prompt

In [73]:
from dspy.teleprompt import BootstrapFewShotWithRandomSearch

# Set up the optimizer: we want to "bootstrap" (i.e., self-generate) 8-shot examples of your program's steps.
# The optimizer will repeat this 10 times (plus some initial attempts) before selecting its best attempt on the devset.

teleprompter = BootstrapFewShotWithRandomSearch(
    metric=llm_metric,
    max_labeled_demos=4,
    max_bootstrapped_demos=1,
    num_candidate_programs=10,
    num_threads=4,
)


Going to sample between 1 and 1 traces per predictor.
Will attempt to bootstrap 10 candidate sets.


In [74]:
answer_generator_compiled = teleprompter.compile(answer_generator, trainset=train, valset=dev)

Average Metric: 13.0 / 40  (32.5): 100%|████████████████| 40/40 [00:00<00:00, 1752.54it/s]


Score: 32.5 for set: [0]
New best sscore: 32.5 for seed -3
Scores so far: [32.5]
Best score: 32.5


Average Metric: 14.0 / 40  (35.0): 100%|████████████████| 40/40 [00:00<00:00, 1503.11it/s]


Score: 35.0 for set: [4]
New best sscore: 35.0 for seed -2
Scores so far: [32.5, 35.0]
Best score: 35.0


  9%|████▋                                               | 9/100 [00:00<00:00, 713.37it/s]


...

Score: 32.5 for set: [4]
Scores so far: [32.5, 35.0, 31.25, 40.0, 35.0, 33.75, 30.0, 32.5, 30.0, 31.25, 30.0, 32.5]
Best score: 40.0
Average of max per entry across top 1 scores: 0.4
Average of max per entry across top 2 scores: 0.5
Average of max per entry across top 3 scores: 0.5125
Average of max per entry across top 5 scores: 0.525
Average of max per entry across top 8 scores: 0.5625
Average of max per entry across top 9999 scores: 0.6


  3%|█▌                                                  | 3/100 [00:00<00:00, 841.55it/s]


Bootstrapped 1 full traces after 4 examples in round 0.


Average Metric: 14.0 / 40  (35.0): 100%|██████████████████| 40/40 [00:15<00:00,  2.57it/s]

Score: 35.0 for set: [4]
Scores so far: [32.5, 35.0, 31.25, 40.0, 35.0, 33.75, 30.0, 32.5, 30.0, 31.25, 30.0, 32.5, 35.0]
Best score: 40.0
Average of max per entry across top 1 scores: 0.4
Average of max per entry across top 2 scores: 0.5
Average of max per entry across top 3 scores: 0.5125
Average of max per entry across top 5 scores: 0.5375
Average of max per entry across top 8 scores: 0.55
Average of max per entry across top 9999 scores: 0.6
13 candidate programs found.


### Test compiled prompt

In [75]:
prediction = answer_generator_compiled(test[0]['question'], test[0]['context'])
print(test[0])
print(prediction)

Example({'question': 'If I overbuild a flipped tile, do I lose out on any points that I would have otherwise scored for that overbuilt tile at the end of the current era? Or do I score the points immediately at the time of overbuilding?', 'context': 'Remove overbuilt Industry tiles from the game, and return them to the box (they will not score VPs). Players do not lose previously gained income or VPs if their Industry tiles are overbuilt.', 'answer': "You lose them. Overbuilt tiles (whether yours or an opponent's) do not score any VP, because they cease to exist."}) (input_keys={'question', 'context'})
Prediction(
    context='Remove overbuilt Industry tiles from the game, and return them to the box (they will not score VPs). Players do not lose previously gained income or VPs if their Industry tiles are overbuilt.',
    answer='No, you do not lose out on any points that you would have otherwise scored for the overbuilt tile at the end of the current era. The points are retained and wi

In [76]:
lm.inspect_history(n=1)




Answer questions from contexts.

---

Context:
On your turn, perform a total of 2 actions.
Exception: During the first round of the Canal Era, each player performs only 1 action. 
Question: So you have always to do 2 actions, exception the first turn of the canal phase, right?
Answer: Yeah, every turn except the very first is exactly two actions, no more and no less.

Context:
Selling requires:
- Connection to a Merchant tile.
- Beer (usually).

Two locations are considered “connected”  to each other if you can trace a route of Link tiles (owned by any player) from one  location to the other
Question: Shrewsbury has only one route into it. Which I believe means that once someone puts a boat on that Canal (or train on the track in the Rail era) only that player will be able to sell Potteries. Is that the correct conclusion?
Answer: In order to sell to a market, you need only have a connection. The links in the connection may have been built by any combinations of players but it is on

'\n\n\nAnswer questions from contexts.\n\n---\n\nContext:\nOn your turn, perform a total of 2 actions.\nException: During the first round of the Canal Era, each player performs only 1 action. \nQuestion: So you have always to do 2 actions, exception the first turn of the canal phase, right?\nAnswer: Yeah, every turn except the very first is exactly two actions, no more and no less.\n\nContext:\nSelling requires:\n- Connection to a Merchant tile.\n- Beer (usually).\n\nTwo locations are considered “connected”  to each other if you can trace a route of Link tiles (owned by any player) from one  location to the other\nQuestion: Shrewsbury has only one route into it. Which I believe means that once someone puts a boat on that Canal (or train on the track in the Rail era) only that player will be able to sell Potteries. Is that the correct conclusion?\nAnswer: In order to sell to a market, you need only have a connection. The links in the connection may have been built by any combinations of

## Evaluate compiled prompt

In [77]:
evaluator = Evaluate(devset=test, num_threads=1, display_progress=True, display_table=5)

In [78]:
# launch evaluation
evaluator(answer_generator_compiled, metric=llm_metric)

Average Metric: 10.0 / 40  (25.0): 100%|██████████████████| 40/40 [01:21<00:00,  2.03s/it]


,question,example_context,example_answer,pred_context,pred_answer,llm_metric
0,"If I overbuild a flipped tile, do I lose out on any points that I would have otherwise scored for that overbuilt tile at the...","Remove overbuilt Industry tiles from the game, and return them to the box (they will not score VPs). Players do not lose previously gained income...","You lose them. Overbuilt tiles (whether yours or an opponent's) do not score any VP, because they cease to exist.","Remove overbuilt Industry tiles from the game, and return them to the box (they will not score VPs). Players do not lose previously gained income...","No, you do not lose out on any points that you would have otherwise scored for the overbuilt tile at the end of the current...",0.0
1,"If you have no tiles on the board and build an industry or link, do you still pay the usual costs?","To Perform the Build Action: 3- The cost of the Industry tile you are building is displayed, to the left of its slot, on your...","Yes, of course. That also includes needing a source of coal.","To Perform the Build Action: 3- The cost of the Industry tile you are building is displayed, to the left of its slot, on your...","Yes, you still pay the usual costs when building an industry or link, even if you have no tiles on the board.",0.0
2,When performing the double-rail action is the beer paid before or after the connections are made?,"In a single Network action, you may build a maximum of 2 rail Links for £15, if you also consume 1 beer. This beer must...",The rules say the second rail link must be connected to an opponent's beer after it is built if you need to use an opponent's...,"In a single Network action, you may build a maximum of 2 rail Links for £15, if you also consume 1 beer. This beer must...","The beer is consumed before the connections are made, as it is a requirement to perform the double-rail action.",0.0
3,If someone has placed a coal industry at some city in their first turn and now i build a canal to that same city in...,Two locations are considered “connected” to each other if you can trace a route of Link tiles (owned by any player) from one location to...,"It is allowed. Note that your opponent can always build with a location card, so your opponent can use a location card or scout for...",Two locations are considered “connected” to each other if you can trace a route of Link tiles (owned by any player) from one location to...,"network, they will have to look for other locations on the board where they can build or expand their network. They may need to focus...",0.0
4,What is the point of building in areas not considered for a gameplay (2/3 P) when it is out of bounds?,"At the end of each era, flipped Industry tiles score VPs. When flipped, they have a black top half and a VP icon in the...","They still work as normal, so they still produce resources, income, points and link points. They are just a bit harder to reach, but they...","At the end of each era, flipped Industry tiles score VPs. When flipped, they have a black top half and a VP icon in the...","Building in out-of-bounds areas may serve strategic purposes, such as preparing for future expansions or influencing other players, even if those areas do not directly...",0.0


25.0

## Re-evaluate original prompt

Note that it's slightly worse

In [79]:
evaluator(answer_generator, metric=llm_metric)

Average Metric: 9.0 / 40  (22.5): 100%|██████████████████| 40/40 [00:00<00:00, 421.63it/s]


,question,example_context,example_answer,pred_context,pred_answer,llm_metric
0,"If I overbuild a flipped tile, do I lose out on any points that I would have otherwise scored for that overbuilt tile at the...","Remove overbuilt Industry tiles from the game, and return them to the box (they will not score VPs). Players do not lose previously gained income...","You lose them. Overbuilt tiles (whether yours or an opponent's) do not score any VP, because they cease to exist.","Remove overbuilt Industry tiles from the game, and return them to the box (they will not score VPs). Players do not lose previously gained income...",You do not score any points for the overbuilt tile at the end of the current era; it is removed and does not contribute to...,✔️ [0.5]
1,"If you have no tiles on the board and build an industry or link, do you still pay the usual costs?","To Perform the Build Action: 3- The cost of the Industry tile you are building is displayed, to the left of its slot, on your...","Yes, of course. That also includes needing a source of coal.","To Perform the Build Action: 3- The cost of the Industry tile you are building is displayed, to the left of its slot, on your...","Yes, you still pay the usual costs.",0.0
2,When performing the double-rail action is the beer paid before or after the connections are made?,"In a single Network action, you may build a maximum of 2 rail Links for £15, if you also consume 1 beer. This beer must...",The rules say the second rail link must be connected to an opponent's beer after it is built if you need to use an opponent's...,"In a single Network action, you may build a maximum of 2 rail Links for £15, if you also consume 1 beer. This beer must...",The beer is consumed before the connections are made.,0.0
3,If someone has placed a coal industry at some city in their first turn and now i build a canal to that same city in...,Two locations are considered “connected” to each other if you can trace a route of Link tiles (owned by any player) from one location to...,"It is allowed. Note that your opponent can always build with a location card, so your opponent can use a location card or scout for...",Two locations are considered “connected” to each other if you can trace a route of Link tiles (owned by any player) from one location to...,"Yes, your actions are allowed. Your opponent can still use the coal industry they built, but they cannot expand their network further if all routes...",0.0
4,What is the point of building in areas not considered for a gameplay (2/3 P) when it is out of bounds?,"At the end of each era, flipped Industry tiles score VPs. When flipped, they have a black top half and a VP icon in the...","They still work as normal, so they still produce resources, income, points and link points. They are just a bit harder to reach, but they...","At the end of each era, flipped Industry tiles score VPs. When flipped, they have a black top half and a VP icon in the...",The point of building in areas not considered for gameplay (2/3 P) when it is out of bounds may be for strategic positioning or future...,0.0


22.5

## Conclusion

In the end, DSPy didn't give me as much of a boost as I wanted. Here are several other things I could try:
- There are several different optimizers (teleprompters) in DSPy. We could try a different optimizer
  - MIPRO is the best one, but it requires several hundred training examples
- max_labeled_demos, max_bootstrapped_demos, and num_candidate_programs are all hyperparameters in BootstrapFewShotWithRandomSearch. We could try different values to see if things improve.
- Most importantly, the computer decides that a prompt is better if the llm_metric function gives better results for the rows in the dev set. We need to verify that we agree with the results from the llm_metric function! If we don't agree with its results then we should change the assessment prompts until we do.